# Set up the tables for week 3, day 1, exercise 1

For the trainer. Run this notebook once per Databricks workspace, before Monday's class. It makes the schema `case_selection` and three tables in it: `returns` (30 rows), `wage_statements` (30 rows) and `exam_outcomes` (12 rows). Each table and each column has a comment. The course made up every row: the names, the addresses and the identifying numbers are not real. To run it again is safe: it replaces the three tables.

1. Open this notebook from the course's Git folder and attach it to compute.
2. In cell 1, set `SCHEMA` to the course catalog's name, a dot, and `case_selection`. The catalog must exist, and you need permission to make a schema in it (`USE CATALOG` and `CREATE SCHEMA` on the catalog).
3. Run cells 1 to 3. Cell 3 prints the columns and comments of each table, and its row count.
4. If the learners cannot see the schema in Catalog Explorer, run cell 4. It gives the group `account users` the privileges the exercise needs. That group holds every user of the account. The privileges are `USE CATALOG` on the catalog, `USE SCHEMA` on the schema, and `SELECT` on the three tables. `BROWSE` alone is not enough: it shows the tables but not their rows. The **Sample Data** tab and cell 1 of the exercise read the rows.
5. Paste the catalog's name in the class chat before the exercise. Learners type `<catalog>.case_selection` in cell 1 of the exercise.

Each statement kind, with the Databricks page that gives its syntax, is in `decisions.md` beside this notebook.

In [ ]:
# 1. Where the tables go: the course catalog's name, a dot, and case_selection. Set this before you run.
SCHEMA = "REPLACE_WITH_COURSE_CATALOG.case_selection"

if SCHEMA.startswith("REPLACE_WITH"):
    raise SystemExit("Set SCHEMA in cell 1 to the course catalog's name, a dot, and case_selection.")

In [ ]:
# 2. Make the schema and the three tables, with their comments and made-up rows.
def statements(schema: str) -> list[str]:
    """Return the SQL that makes the schema, the three tables, their comments and their rows.

    Args:
        schema: the catalog's name, a dot, and the schema's name.

    Returns:
        One SQL statement per item, in the order they run.
    """
    made_up = "Made up by the course for week 3, day 1, exercise 1; no row is a real taxpayer."
    sql = [
        f"CREATE SCHEMA IF NOT EXISTS {schema} COMMENT 'Tables for the case-selection exercises of week 3. {made_up}'",
        f"""CREATE OR REPLACE TABLE {schema}.returns (
  tin STRING COMMENT 'the taxpayer identification number, which joins the three tables',
  taxpayer_name STRING COMMENT 'the name of the taxpayer',
  street_address STRING COMMENT 'the street address of the taxpayer',
  wages_amt INT COMMENT 'the wages the taxpayer reported on the return',
  total_income_amt INT COMMENT 'the total income the taxpayer reported on the return'
) COMMENT 'One row per return filed this year. {made_up}'""",
        f"""CREATE OR REPLACE TABLE {schema}.wage_statements (
  tin STRING COMMENT 'the taxpayer identification number, which joins the three tables',
  w2_wages_amt INT COMMENT 'the wages the employer reported on the W-2 for the same year'
) COMMENT 'One row per W-2 wage statement an employer sent for this year. {made_up}'""",
        f"""CREATE OR REPLACE TABLE {schema}.exam_outcomes (
  tin STRING COMMENT 'the taxpayer identification number, which joins the three tables',
  exam_changed_tax BOOLEAN COMMENT 'for a return examined in the past: did the examination change the tax, yes or no',
  exam_picked_by STRING COMMENT 'for a return examined in the past: how it was picked, random sample or score'
) COMMENT 'One row per past examination of a return. {made_up}'""",
    ]
    returns, wages, exams = [], [], []
    for i in range(1, 31):
        tin = f"000-00-{i:04d}"
        pay = 30000 + 2750 * i
        returns.append(f"('{tin}', 'Made-up Taxpayer {i:02d}', '{i} Example Street, Sample City', {pay}, {pay + 1200 * (i % 5)})")
        wages.append(f"('{tin}', {pay + (4000 if i % 6 == 0 else 0)})")
        if i <= 12:
            picked = "random sample" if i % 4 == 0 else "score"
            exams.append(f"('{tin}', {'true' if i % 3 else 'false'}, '{picked}')")
    sql.append(f"INSERT INTO {schema}.returns VALUES\n" + ",\n".join(returns))
    sql.append(f"INSERT INTO {schema}.wage_statements VALUES\n" + ",\n".join(wages))
    sql.append(f"INSERT INTO {schema}.exam_outcomes VALUES\n" + ",\n".join(exams))
    return sql


for statement in statements(SCHEMA):
    spark.sql(statement)
print("Made", SCHEMA, "with returns, wage_statements and exam_outcomes. Run cell 3.")

In [ ]:
# 3. Read back what a learner sees: each column's comment, and each table's row count.
for table in ["returns", "wage_statements", "exam_outcomes"]:
    count = spark.sql(f"SELECT count(*) AS n FROM {SCHEMA}.{table}").collect()[0].n
    print(f"{SCHEMA}.{table}: {count} rows")
    for row in spark.sql(f"DESCRIBE TABLE {SCHEMA}.{table}").collect():
        print(f"  {row.col_name:<17} {row.data_type:<8} {row.comment}")

In [ ]:
# 4. Only if learners cannot see the schema: let every user of the account read it.
catalog = SCHEMA.split(".")[0]
for statement in [
    f"GRANT USE CATALOG ON CATALOG {catalog} TO `account users`",
    f"GRANT USE SCHEMA ON SCHEMA {SCHEMA} TO `account users`",
    f"GRANT SELECT ON TABLE {SCHEMA}.returns TO `account users`",
    f"GRANT SELECT ON TABLE {SCHEMA}.wage_statements TO `account users`",
    f"GRANT SELECT ON TABLE {SCHEMA}.exam_outcomes TO `account users`",
]:
    spark.sql(statement)
print("Granted USE CATALOG, USE SCHEMA and SELECT on", SCHEMA, "to account users.")